# 단풍도서관 RAG · 현행 문서와 과거 안내문 비교

## 1단계 · 현행 문서로 답하기

가상의 단풍도서관에서 **일반 도서의 기본 대출 기간은 며칠인가요?** 현행 안내문 열 개를 문단 단위로 나누고, FAISS가 찾은 상위 청크 세 개를 한 번에 생성 모델에 전달한다. 한국어 질문, 검색 결과, 프롬프트, 답변을 함께 기록해 이후 실험의 기준으로 쓴다.

Python 3 환경에서 필요한 경우 아래 설치 명령의 주석을 풀고, 위에서부터 모든 셀을 실행한다. 처음 실행할 때 공개 모델을 내려받는다. 캐시에 모델이 있으면 CPU에서도 오프라인으로 실행할 수 있다.

In [1]:
# %pip install faiss-cpu scikit-learn numpy torch "transformers>=4.51"

## 문서와 청크

각 문서는 다른 상황에서 독자가 접하는 안내문이다. 빈 줄로 나뉜 문단 하나를 청크 하나로 사용한다. 청크 ID 앞부분은 원문 문서 ID이고 뒷부분은 그 안의 문단 번호다.

In [2]:
documents = {
    "이용규정": """일반 도서의 기본 대출 기간은 14일입니다.

참고 도서는 도서관 안에서만 열람할 수 있습니다.""",
    "회원안내": """회원은 일반 도서를 14일 동안 빌릴 수 있습니다.

도서관 회원증을 신청하려면 안내 데스크에 사진이 있는 신분증을 제시하세요.""",
    "대출영수증": """일반 도서의 대출 기한은 대출일로부터 14일입니다.

대출 영수증에는 빌린 책의 제목과 반납 예정일이 표시됩니다.""",
    "반납안내": """야외 도서 반납함은 도서관 정문 바로 옆에 있습니다.

대형 자료는 안내 데스크에 직접 반납해 주세요.""",
    "운영시간": """단풍도서관의 주말 개관 시간은 오전 10시입니다.

평일에는 오전 9시에 문을 열고 오후 8시에 닫습니다.""",
    "예약안내": """다른 회원이 빌린 책은 도서관 누리집에서 예약할 수 있습니다.

예약한 책이 도착하면 문자로 안내합니다.""",
    "연장안내": """예약 대기자가 없는 책은 한 번 연장할 수 있습니다.

연장은 반납 예정일 전에 누리집의 내 서재에서 신청하세요.""",
    "열람실안내": """조용한 열람실은 본관 2층에 있습니다.

열람실 좌석은 입구의 단말기에서 선택하세요.""",
    "복사안내": """흑백 복사기는 본관 1층 자료실에 있습니다.

흑백 복사 요금은 한 면에 50원입니다.""",
    "분실물안내": """도서관에서 잃어버린 물건은 안내 데스크에 문의하세요.

접수한 분실물은 한 달 동안 보관합니다.""",
}

def split_documents(documents):
    return [
        {"id": f"{doc_id}/{number}", "doc_id": doc_id, "text": paragraph.strip()}
        for doc_id, text in documents.items()
        for number, paragraph in enumerate(text.split("\n\n"), start=1)
        if paragraph.strip()
    ]

chunks = split_documents(documents)
print(f"문서 {len(documents)}개 → 청크 {len(chunks)}개")
for chunk in chunks:
    print(f"{chunk['id']} · {chunk['text']}")

문서 10개 → 청크 20개
이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
이용규정/2 · 참고 도서는 도서관 안에서만 열람할 수 있습니다.
회원안내/1 · 회원은 일반 도서를 14일 동안 빌릴 수 있습니다.
회원안내/2 · 도서관 회원증을 신청하려면 안내 데스크에 사진이 있는 신분증을 제시하세요.
대출영수증/1 · 일반 도서의 대출 기한은 대출일로부터 14일입니다.
대출영수증/2 · 대출 영수증에는 빌린 책의 제목과 반납 예정일이 표시됩니다.
반납안내/1 · 야외 도서 반납함은 도서관 정문 바로 옆에 있습니다.
반납안내/2 · 대형 자료는 안내 데스크에 직접 반납해 주세요.
운영시간/1 · 단풍도서관의 주말 개관 시간은 오전 10시입니다.
운영시간/2 · 평일에는 오전 9시에 문을 열고 오후 8시에 닫습니다.
예약안내/1 · 다른 회원이 빌린 책은 도서관 누리집에서 예약할 수 있습니다.
예약안내/2 · 예약한 책이 도착하면 문자로 안내합니다.
연장안내/1 · 예약 대기자가 없는 책은 한 번 연장할 수 있습니다.
연장안내/2 · 연장은 반납 예정일 전에 누리집의 내 서재에서 신청하세요.
열람실안내/1 · 조용한 열람실은 본관 2층에 있습니다.
열람실안내/2 · 열람실 좌석은 입구의 단말기에서 선택하세요.
복사안내/1 · 흑백 복사기는 본관 1층 자료실에 있습니다.
복사안내/2 · 흑백 복사 요금은 한 면에 50원입니다.
분실물안내/1 · 도서관에서 잃어버린 물건은 안내 데스크에 문의하세요.
분실물안내/2 · 접수한 분실물은 한 달 동안 보관합니다.


## FAISS 검색

조사가 붙어도 ‘반납함’과 ‘반납함은’의 공통 부분을 찾을 수 있도록 어절 안에서 3~5글자씩 겹쳐 읽는다. 어절 양끝의 공백도 글자 수에 포함한다. 질문과 청크를 이 문자 조각의 TF-IDF 벡터로 표현하고 L2 정규화한다. FAISS가 계산한 내적은 코사인 유사도이며, 점수가 0보다 큰 청크를 최대 $k$개 반환한다.

In [3]:
import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

class PassageIndex:
    def __init__(self, chunks, *, vectorizer=None):
        chunks = list(chunks)
        if vectorizer is None:
            vectorizer = TfidfVectorizer(
                analyzer="char_wb", ngram_range=(3, 5), dtype=np.float32, norm="l2"
            )
            vectorizer.fit([chunk["text"] for chunk in chunks])
        self.vectorizer = vectorizer
        self.index = faiss.IndexFlatIP(len(self.vectorizer.vocabulary_))
        self.chunks = []
        self.add_chunks(chunks)

    def add_chunks(self, chunks):
        batch = list(chunks)
        if not batch:
            return
        new_ids = {chunk["id"] for chunk in batch}
        if len(new_ids) != len(batch) or any(chunk["id"] in new_ids for chunk in self.chunks):
            raise ValueError("같은 청크 ID가 두 번 등록됩니다.")
        vectors = self.vectorizer.transform([chunk["text"] for chunk in batch])
        self.index.add(vectors.toarray())
        self.chunks.extend(batch)

    def search(self, question, k=3):
        if k <= 0 or self.index.ntotal == 0:
            return []
        query = self.vectorizer.transform([question])
        if query.nnz == 0:
            return []
        scores, positions = self.index.search(query.toarray(), min(k, self.index.ntotal))
        return [
            {**self.chunks[position], "score": float(score)}
            for score, position in zip(scores[0], positions[0])
            if score > 0
        ]

retriever = PassageIndex(chunks)
question = "일반 도서의 기본 대출 기간은 며칠인가요?"
hits = retriever.search(question, k=3)

print(question)
for rank, hit in enumerate(hits, start=1):
    print(f"{rank}. [{hit['score']:.3f}] {hit['id']} · {hit['text']}")

일반 도서의 기본 대출 기간은 며칠인가요?
1. [0.779] 이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
2. [0.329] 대출영수증/1 · 일반 도서의 대출 기한은 대출일로부터 14일입니다.
3. [0.103] 회원안내/1 · 회원은 일반 도서를 14일 동안 빌릴 수 있습니다.


## 검색한 근거로 답변 생성

상위 청크들을 하나의 한국어 프롬프트에 담아 Qwen3-0.6B를 한 번 호출한다. 같은 조건에서 결과를 비교할 수 있도록 모델 버전을 고정하고, 비사고 모드에서 매번 확률이 가장 높은 토큰을 선택한다. 검색 결과가 비면 모델 호출을 건너뛴다. [Qwen3-0.6B 모델 카드](https://huggingface.co/Qwen/Qwen3-0.6B)

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as model_logging

model_logging.disable_progress_bar()
torch.set_num_threads(min(torch.get_num_threads(), 4))
model_id = "Qwen/Qwen3-0.6B"
revision = "c1899de289a04d12100db370d81485cdf75e47ca"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=revision, use_safetensors=True
).eval()

def make_prompt(question, passages):
    context = "\n".join(passage["text"] for passage in passages)
    return (
        f"다음은 도서관 안내문입니다.\n\n{context}\n\n질문: {question}\n"
        "안내문에서 질문의 답을 찾아 한국어로 한 문장으로 답하세요."
    )

@torch.inference_mode()
def answer_from(question, passages):
    if not passages:
        return "근거 없음"
    messages = [{"role": "user", "content": make_prompt(question, passages)}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt", return_dict=True,
    )
    output = model.generate(**inputs, do_sample=False, max_new_tokens=48)
    answer_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

print(make_prompt(question, hits))
answer = answer_from(question, hits)
print(f"\n모델 답변 → {answer}")

다음은 도서관 안내문입니다.

일반 도서의 기본 대출 기간은 14일입니다.
일반 도서의 대출 기한은 대출일로부터 14일입니다.
회원은 일반 도서를 14일 동안 빌릴 수 있습니다.

질문: 일반 도서의 기본 대출 기간은 며칠인가요?
안내문에서 질문의 답을 찾아 한국어로 한 문장으로 답하세요.



모델 답변 → 일반 도서의 기본 대출 기간은 14일입니다.


## 기준 결과 확인

대출 기간을 뒷받침하는 청크 세 개가 모두 검색됐는지, 생성 답변에 올바른 기간과 반납함 위치가 담겼는지 검사한다. 문서에 없는 망원경 질문은 검색 결과가 비어 ‘근거 없음’으로 처리되는지도 확인한다. 추가한 운영시간 안내문으로 주말 개관 시간도 확인한다.

In [5]:
assert len(documents) == 10
assert retriever.index.ntotal == len(chunks) == 20
assert hits[0]["id"] == "이용규정/1"
assert {hit["id"] for hit in hits} == {"이용규정/1", "회원안내/1", "대출영수증/1"}
assert "14일" in answer.replace(" ", "") or "열나흘" in answer, answer

location_question = "야외 도서 반납함은 어디에 있나요?"
location_hits = retriever.search(location_question)
location_answer = answer_from(location_question, location_hits)
print(f"{location_question} → {location_answer} [{location_hits[0]['id']}]")
assert location_hits[0]["id"] == "반납안내/1"
assert "정문" in location_answer and "옆" in location_answer, location_answer

unknown_question = "망원경이 있나요?"
unknown_hits = retriever.search(unknown_question)
unknown_answer = answer_from(unknown_question, unknown_hits)
print(f"{unknown_question} → {unknown_answer} (검색 {len(unknown_hits)}건)")
assert not unknown_hits and unknown_answer == "근거 없음"

hours_question = "단풍도서관의 주말 개관 시간은 언제인가요?"
hours_hits = retriever.search(hours_question)
hours_answer = answer_from(hours_question, hours_hits)
print(f"{hours_question} → {hours_answer} [{hours_hits[0]['id']}]")
assert hours_hits[0]["id"] == "운영시간/1"
assert "오전10시" in hours_answer.replace(" ", ""), hours_answer

야외 도서 반납함은 어디에 있나요? → 야외 도서 반납함은 도서관 정문 바로 옆에 있습니다. [반납안내/1]
망원경이 있나요? → 근거 없음 (검색 0건)


단풍도서관의 주말 개관 시간은 언제인가요? → 단풍도서관의 주말 개관 시간은 오전 10시입니다. [운영시간/1]


## 2단계 · 과거 안내문이 현행 자료에 섞이면 어떻게 될까?

이 실험에서 ‘오염 문서’는 대출 기간이 7일이던 2021년의 가상 안내문이다. 당시에는 맞지만 현재 일반 도서 대출 기간인 14일과 충돌한다. 같은 현행 청크 20개로 별도 인덱스를 만든 뒤 보관본을 추가한다. TF-IDF 어휘와 IDF는 현행 자료로 학습한 상태에서 고정하므로 기존 청크의 점수를 삽입 전후에 직접 비교할 수 있다.

대출 기간 질문으로 상충하는 근거가 검색되는지 확인하고, 규정 변경과 관계없는 주말 개관 시간 질문도 같은 조건으로 비교한다.

In [6]:
archived_documents = {
    "보관본-2021": "2021년 이용 안내에 따르면 일반 도서의 대출 기간은 7일입니다."
}
archived_chunks = split_documents(archived_documents)
mixed_retriever = PassageIndex(chunks, vectorizer=retriever.vectorizer)
mixed_retriever.add_chunks(archived_chunks)

print(f"현행 자료: 문서 {len(documents)}개, 청크 {retriever.index.ntotal}개")
print(f"보관본 포함: 문서 {len(documents) + len(archived_documents)}개, 청크 {mixed_retriever.index.ntotal}개")
for chunk in archived_chunks:
    print(f"추가된 청크 · {chunk['id']} · {chunk['text']}")

assert retriever.index.ntotal == 20 and mixed_retriever.index.ntotal == 21

현행 자료: 문서 10개, 청크 20개
보관본 포함: 문서 11개, 청크 21개
추가된 청크 · 보관본-2021/1 · 2021년 이용 안내에 따르면 일반 도서의 대출 기간은 7일입니다.


In [7]:
questions = [
    (question, hits, answer),
    (hours_question, hours_hits, hours_answer),
]
archive_ids = {chunk["id"] for chunk in archived_chunks}

mixed_results = {}
for query, before_hits, before_answer in questions:
    after_hits = mixed_retriever.search(query, k=3)
    after_answer = answer_from(query, after_hits)
    mixed_results[query] = after_hits, after_answer

    print(f"질문 · {query}")
    for label, passages, response in [
        ("삽입 전", before_hits, before_answer),
        ("삽입 후", after_hits, after_answer),
    ]:
        print(label)
        for rank, passage in enumerate(passages, start=1):
            print(f"  {rank}위 [{passage['score']:.3f}] {passage['id']} · {passage['text']}")
        print(f"  답변 → {response}")
    archive_count = sum(hit["id"] in archive_ids for hit in after_hits)
    print(f"삽입 후 상위 {len(after_hits)}개 중 보관본 {archive_count}개")
    print("답변 문구 동일" if after_answer == before_answer else "답변 문구 변경")
    print()

질문 · 일반 도서의 기본 대출 기간은 며칠인가요?
삽입 전
  1위 [0.779] 이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
  2위 [0.329] 대출영수증/1 · 일반 도서의 대출 기한은 대출일로부터 14일입니다.
  3위 [0.103] 회원안내/1 · 회원은 일반 도서를 14일 동안 빌릴 수 있습니다.
  답변 → 일반 도서의 기본 대출 기간은 14일입니다.
삽입 후
  1위 [0.779] 이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
  2위 [0.778] 보관본-2021/1 · 2021년 이용 안내에 따르면 일반 도서의 대출 기간은 7일입니다.
  3위 [0.329] 대출영수증/1 · 일반 도서의 대출 기한은 대출일로부터 14일입니다.
  답변 → 일반 도서의 기본 대출 기간은 14일입니다.
삽입 후 상위 3개 중 보관본 1개
답변 문구 동일



질문 · 단풍도서관의 주말 개관 시간은 언제인가요?
삽입 전
  1위 [0.785] 운영시간/1 · 단풍도서관의 주말 개관 시간은 오전 10시입니다.
  2위 [0.029] 이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
  3위 [0.014] 반납안내/1 · 야외 도서 반납함은 도서관 정문 바로 옆에 있습니다.
  답변 → 단풍도서관의 주말 개관 시간은 오전 10시입니다.
삽입 후
  1위 [0.785] 운영시간/1 · 단풍도서관의 주말 개관 시간은 오전 10시입니다.
  2위 [0.035] 보관본-2021/1 · 2021년 이용 안내에 따르면 일반 도서의 대출 기간은 7일입니다.
  3위 [0.029] 이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
  답변 → 단풍도서관의 주말 개관 시간은 오전 10시입니다.
삽입 후 상위 3개 중 보관본 1개
답변 문구 동일



### 관찰 결과

대출 기간 질문의 상위 3개 청크에는 처음에 모두 ‘14일’이 들어 있었다. 보관본을 넣자 ‘7일’이 적힌 문단이 2위(0.778)에 들어와 회원안내 문단을 밀어냈다. 현행 이용규정은 1위(0.779)였고 모델은 두 경우 모두 ‘14일’이라고 답했다.

주말 개관 시간 질문에서도 보관본이 2위에 들어왔지만 유사도는 0.035였다. 1위는 운영시간 안내(0.785)였고 답변은 두 경우 모두 ‘오전 10시’였다.

## 3단계 · 적용 기간이 끝난 문서 정제하기

단풍도서관이 문서별 적용 시작일과 종료일을 별도 대장으로 관리한다고 가정한다. 아래 날짜는 가상의 등록 정보다. 실행 날짜가 달라져도 같은 자료를 비교하도록 기준일을 **2026년 11월 23일**로 고정한다.

시작일과 종료일 당일은 적용 기간에 포함한다. 종료일이 비어 있으면 계속 적용한다. 대장에 없는 문서와 적용 기간 밖의 문서는 제외 사유를 기록한다. 정제 입력은 2단계에서 만든 혼합 인덱스의 전체 청크다.

In [8]:
from datetime import date

as_of = date(2026, 11, 23)
# 문서 ID별 (적용 시작일, 마지막 적용일)
validity_periods = {
    "이용규정": (date(2022, 1, 1), None),
    "회원안내": (date(2022, 1, 1), None),
    "대출영수증": (date(2022, 1, 1), None),
    "반납안내": (date(2022, 3, 1), None),
    "운영시간": (date(2023, 1, 1), None),
    "예약안내": (date(2023, 1, 1), None),
    "연장안내": (date(2023, 1, 1), None),
    "열람실안내": (date(2024, 1, 1), None),
    "복사안내": (date(2024, 1, 1), None),
    "분실물안내": (date(2024, 1, 1), None),
    "보관본-2021": (date(2021, 1, 1), date(2021, 12, 31)),
}

def exclusion_reason(period, on_date):
    if period is None:
        return "적용 기간 미등록"
    start, end = period
    if end is not None and end < start:
        return "종료일이 시작일보다 빠름"
    if on_date < start:
        return f"{start}부터 적용 예정"
    if end is not None and on_date > end:
        return f"{end}에 적용 종료"
    return None

In [9]:
mixed_chunks = mixed_retriever.chunks
doc_ids = {chunk["doc_id"] for chunk in mixed_chunks}
excluded = {}
for doc_id in sorted(doc_ids):
    reason = exclusion_reason(validity_periods.get(doc_id), as_of)
    if reason:
        excluded[doc_id] = reason

current_chunks = [chunk for chunk in mixed_chunks if chunk["doc_id"] not in excluded]
cleaned_retriever = PassageIndex(current_chunks, vectorizer=mixed_retriever.vectorizer)

print(f"기준일 {as_of} · 문서 {len(doc_ids)}개 → {len(doc_ids) - len(excluded)}개")
print(f"FAISS 청크 {mixed_retriever.index.ntotal}개 → {cleaned_retriever.index.ntotal}개")
for doc_id, reason in excluded.items():
    print(f"제외 문서 · {doc_id} · {reason}")

assert {chunk["id"] for chunk in cleaned_retriever.chunks} == {chunk["id"] for chunk in chunks}
assert cleaned_retriever.index.ntotal == len(chunks)

기준일 2026-11-23 · 문서 11개 → 10개
FAISS 청크 21개 → 20개
제외 문서 · 보관본-2021 · 2021-12-31에 적용 종료


### 같은 질문으로 정제 효과 확인

현행 자료만 넣은 기준 상태, 보관본이 섞인 상태, 정제한 상태를 비교한다. 세 조건에서 같은 TF-IDF 어휘·IDF, 상위 청크 수(3개), 모델과 프롬프트를 사용한다. 각 행에서 검색된 청크는 순위순이고 괄호 안은 코사인 유사도다.

In [10]:
for query, baseline_hits, baseline_answer in questions:
    mixed_hits, mixed_answer = mixed_results[query]
    cleaned_hits = cleaned_retriever.search(query, k=3)
    cleaned_answer = answer_from(query, cleaned_hits)
    print(f"질문 · {query}")
    for label, passages, response in [
        ("현행 기준", baseline_hits, baseline_answer),
        ("보관본 삽입 후", mixed_hits, mixed_answer),
        ("정제 후", cleaned_hits, cleaned_answer),
    ]:
        ranking = " → ".join(f"{p['id']} ({p['score']:.3f})" for p in passages)
        print(f"  {label} · {ranking}")
        print(f"    {response}")
    print()

    assert [p["id"] for p in cleaned_hits] == [p["id"] for p in baseline_hits]
    np.testing.assert_allclose([p["score"] for p in cleaned_hits], [p["score"] for p in baseline_hits])
    assert cleaned_answer == baseline_answer

질문 · 일반 도서의 기본 대출 기간은 며칠인가요?
  현행 기준 · 이용규정/1 (0.779) → 대출영수증/1 (0.329) → 회원안내/1 (0.103)
    일반 도서의 기본 대출 기간은 14일입니다.
  보관본 삽입 후 · 이용규정/1 (0.779) → 보관본-2021/1 (0.778) → 대출영수증/1 (0.329)
    일반 도서의 기본 대출 기간은 14일입니다.
  정제 후 · 이용규정/1 (0.779) → 대출영수증/1 (0.329) → 회원안내/1 (0.103)
    일반 도서의 기본 대출 기간은 14일입니다.



질문 · 단풍도서관의 주말 개관 시간은 언제인가요?
  현행 기준 · 운영시간/1 (0.785) → 이용규정/1 (0.029) → 반납안내/1 (0.014)
    단풍도서관의 주말 개관 시간은 오전 10시입니다.
  보관본 삽입 후 · 운영시간/1 (0.785) → 보관본-2021/1 (0.035) → 이용규정/1 (0.029)
    단풍도서관의 주말 개관 시간은 오전 10시입니다.
  정제 후 · 운영시간/1 (0.785) → 이용규정/1 (0.029) → 반납안내/1 (0.014)
    단풍도서관의 주말 개관 시간은 오전 10시입니다.



### 관찰 결과

대출 기간 질문에서 2위였던 보관본이 빠지고 회원안내가 다시 상위 3개에 포함됐다. 세 근거가 모두 현행 대출 기간인 14일을 제시했다. 주말 개관 시간 질문에서도 정제 후 상위 3개 청크와 유사도 점수가 처음의 현행 기준과 일치했다.

두 질문 모두 보관본이 섞인 상태에서도 모델이 올바르게 답했다. 이번 실행에서 정제의 효과는 검색 근거에서 만료된 문서를 제거한 것으로 확인했다.